<a href="https://colab.research.google.com/github/diazcas2/mIArmario/blob/main/MODULO3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install google-search-results --quiet

  Preparing metadata (setup.py) ... done


In [47]:
!pip install google-genai --quiet

In [54]:
import google.generativeai as genai
from serpapi import GoogleSearch
import requests
import json
import re
import time
from google.colab import userdata

# Pon tu api key de gemini aquí
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')   # Google Cloud → Custom Search API
SEARCH_ENGINE_ID = userdata.get('SEARCH_ENGINE_ID') # programmablesearchengine.google.com → cx

SERPAPI_KEY = userdata.get('SERPAPI_KEY')  # serpapi.com → dashboard → API Key

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

In [50]:


# Paso 1: Gemini genera las queries de búsqueda
def generar_queries(outfit_json):
    prompt = f"""
      Dado este outfit:
      {json.dumps(outfit_json["outfit"], ensure_ascii=False, indent=2)}

      Genera 3 búsquedas para encontrar prendas complementarias en tiendas online.
      Cada búsqueda debe ser específica: incluye color, tipo de prenda y "comprar".
      Piensa qué le falta al outfit para completarlo.

      Responde ÚNICAMENTE con este JSON, sin texto adicional ni bloques markdown:
      {{
        "queries": [
          "zapatos negros formales hombre comprar zara",
          "cinturón cuero negro formal comprar",
          "corbata seda azul marino comprar mango"
        ]
      }}
      """
    respuesta = model.generate_content(prompt)
    texto = re.sub(r"```json|```", "", respuesta.text).strip()
    return json.loads(texto)["queries"]

def buscar_en_google(query, num_resultados=5):
  params = {
    "engine": "google",       # google normal, no google_shopping
    "q": query + " comprar",
    "api_key": SERPAPI_KEY,
    "gl": "es",
    "hl": "es",
    "num": num_resultados
  }
  search = GoogleSearch(params)
  data = search.get_dict()

  resultados = []
  for r in data.get("organic_results", [])[:num_resultados]:
      enlace = r.get("link", "")
      # Filtrar solo tiendas de ropa conocidas
      tiendas_validas = ["zara", "mango", "hm", "asos", "zalando",
                        "pull", "massimo", "corteingles", "bershka"]
      if any(t in enlace.lower() for t in tiendas_validas):
          resultados.append({
              "titulo": r.get("title", ""),
              "enlace": enlace,
              "descripcion": r.get("snippet", ""),
              "tienda": r.get("displayed_link", ""),
          })

  return resultados


def llamar_gemini(prompt, reintentos=3):
  for intento in range(reintentos):
      try:
          respuesta = model.generate_content(prompt)
          return respuesta.text
      except Exception as e:
          if "429" in str(e):
              espera = 35 * (intento + 1)
              print(f"Límite alcanzado, esperando {espera}s...")
              time.sleep(espera)
          else:
              raise e
  raise Exception("Se agotaron los reintentos")

In [51]:
def modulo3_seleccion_outfit(armario_json, instruccion_usuario, contexto={}):

    # Primero buscamos en SerpAPI sin necesitar Gemini
    queries = [
        f"{instruccion_usuario} zapatos comprar",
        f"{instruccion_usuario} complementos comprar",
        f"{instruccion_usuario} accesorios moda comprar"
    ]

    todos_resultados = []
    for q in queries:
        todos_resultados.extend(buscar_en_google(q))

    # Una sola llamada a Gemini que hace todo
    prompt = f"""
      Eres un estilista experto. Dado este armario y contexto, haz TODO en una sola respuesta.

      ## ARMARIO
      {json.dumps(armario_json, ensure_ascii=False, indent=2)}

      ## INSTRUCCIÓN
      {instruccion_usuario}

      ## CONTEXTO
      Temperatura: {contexto.get('temperatura', 'no especificada')}
      Ocasión: {contexto.get('ocasion', 'no especificada')}

      ## RESULTADOS DE TIENDAS ONLINE (usa SOLO estos enlaces)
      {json.dumps(todos_resultados, ensure_ascii=False, indent=2)}

      Responde ÚNICAMENTE con este JSON:
      {{
        "outfit": {{
          "prendas": [{{"id": "id", "tipo": "tipo", "color": "color"}}],
          "estilo": "formal/casual/sport",
          "ocasion": "{contexto.get('ocasion', '')}",
          "temperatura": {contexto.get('temperatura', 0)}
        }},
        "prompt_imagen": "descripción en inglés para Stable Diffusion, full body, fashion photography, studio lighting",
        "referencias_tiendas": [
          {{
            "prenda": "título exacto de los resultados",
            "tienda": "tienda exacta",
            "enlace": "enlace exacto de los resultados",
            "por_que": "razón"
          }}
        ]
      }}
      """
    texto = llamar_gemini(prompt)
    texto = re.sub(r"```json|```", "", texto).strip()
    return json.loads(texto)

armario = {
    "armario": [
        {
            "id": "prenda_001",
            "tipo": "camisa",
            "color": "blanco",
            "estampado": "liso",
            "material": "algodón",
            "formalidad": "formal",
            "temporada": ["primavera", "verano", "otoño"],
            "imagen_path": "fotos/camisa_blanca_001.jpg"
        },
        {
            "id": "prenda_002",
            "tipo": "pantalón",
            "color": "negro",
            "estampado": "liso",
            "material": "lana",
            "formalidad": "formal",
            "temporada": ["otoño", "invierno"],
            "imagen_path": "fotos/pantalon_negro_002.jpg"
        }
    ]
}

resultado = modulo3_seleccion_outfit(
    armario_json=armario,
    instruccion_usuario="Quiero ir a una cena formal esta noche",
    contexto={"temperatura": 18, "ocasion": "cena"}
)

print(json.dumps(resultado, ensure_ascii=False, indent=2))

Límite alcanzado, esperando 35s...


Límite alcanzado, esperando 70s...


KeyboardInterrupt: 

In [25]:
r = requests.get(
    "https://www.googleapis.com/customsearch/v1",
    params={"key": GOOGLE_API_KEY, "cx": SEARCH_ENGINE_ID, "q": "test"}
)

print("status:", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

status: 403
{
  "error": {
    "code": 403,
    "message": "This project does not have the access to Custom Search JSON API.",
    "errors": [
      {
        "message": "This project does not have the access to Custom Search JSON API.",
        "domain": "global",
        "reason": "forbidden"
      }
    ],
    "status": "PERMISSION_DENIED"
  }
}


In [33]:
resultados = buscar_en_google("zapatos negros formales zara")
print(f"Resultados: {len(resultados)}")
for r in resultados:
    print(r["enlace"])

Resultados: 5
https://www.zara.com/es/es/hombre-zapatos-negros-l2566.html
https://www.zara.com/es/es/mujer-zapatos-negros-l2252.html
https://www.zara.com/es/es/hombre-zapatos-zapatos-l796.html
https://www.zara.com/es/es/hombre-zapatos-noche-l2569.html
https://www.zara.com/es/es/hombre-zapatos-l769.html


In [ ]:
'''# Paso 3: Gemini filtra y elige los mejores enlaces
def filtrar_resultados(outfit_json, todos_los_resultados):

    # Primero comprobamos que SerpAPI devolvió resultados
    print(f"Total resultados de SerpAPI: {len(todos_los_resultados)}")
    print(json.dumps(todos_los_resultados[:3], ensure_ascii=False, indent=2))  # debug

    prompt = f"""
      Tenemos este outfit:
      {json.dumps(outfit_json["outfit"], ensure_ascii=False, indent=2)}

      Estos son los resultados REALES de tiendas online. Usa ÚNICAMENTE estos resultados,
      no inventes ningún producto ni enlace:
      {json.dumps(todos_los_resultados, ensure_ascii=False, indent=2)}

      Selecciona los 3 mejores productos de la lista anterior que complementen el outfit.
      IMPORTANTE: el campo "enlace" debe ser EXACTAMENTE el que aparece en los resultados,
      no lo modifiques ni lo inventes. Si un resultado no tiene enlace, no lo incluyas.

      Responde ÚNICAMENTE con este JSON, sin texto adicional ni bloques markdown:
      {{
        "referencias_tiendas": [
          {{
            "prenda": "título exacto del producto de los resultados",
            "tienda": "tienda exacta de los resultados",
            "enlace": "enlace exacto de los resultados",
            "precio": "precio exacto de los resultados",
            "por_que": "por qué complementa el outfit"
          }}
        ]
      }}
      """
    respuesta = model.generate_content(prompt)
    texto = re.sub(r"```json|```", "", respuesta.text).strip()
    return json.loads(texto)["referencias_tiendas"]'''

    # Paso 2: Buscar en Google
'''def buscar_en_google(query, num_resultados=5):
    url = "https://www.googleapis.com/customsearch/v1"
    params = {
        "key": GOOGLE_API_KEY,
        "cx": SEARCH_ENGINE_ID,
        "q": query,
        "num": num_resultados,
    }
    respuesta = requests.get(url, params=params)
    data = respuesta.json()

    resultados = []
    for item in data.get("items", []):
        resultados.append({
            "titulo": item.get("title", ""),
            "enlace": item.get("link", ""),
            "descripcion": item.get("snippet", ""),
        })
    return resultados'''